In [ ]:
from constant import openalex_path
import polars as pl
from constant import data_path

doi_lf = (
    pl.scan_parquet(f"{openalex_path}/works/doi/*")
    .with_columns(
        pl.col("doi")
        .str.to_lowercase()
        .str.replace_all(r"https://doi.org/", "", literal=True)
    )
    .drop_nulls()
)
source_lf = pl.scan_parquet(f"{openalex_path}/works/primary_location_id/*")
year_lf = pl.scan_parquet(f"{openalex_path}/works/publication_year/*")
ref_lf = pl.scan_parquet(f"{openalex_path}/works/referenced_works/*")

In [ ]:
doi_arxiv = (
    pl.scan_parquet("data/arxiv_id_doi_*.parquet.zstd")
    .with_columns(
        pl.col("doi")
        .str.to_lowercase()
        .str.replace_all(r"https://doi.org/", "", literal=True)
    )
    .drop_nulls("doi")
    .join(doi_lf, on="doi", how="anti")
)
    
doi_arxiv.sink_parquet("data/doi_arxiv_missing_in_openalex.parquet.zstd")

In [ ]:
doi_missing_in_oav1 = pl.scan_parquet("data/doi_arxiv_missing_in_openalex.parquet.zstd")
doi_lf_enhanced_arxiv = (
    pl.concat([doi_lf, doi_missing_in_oav1], how="vertical")
)

In [ ]:
## source ID is not reliable. OpenAlex UI shows Works count: 14,290 for SocArXiv but https://openalex.org/works?filter=primary_location.source.id:s4306401238 returns only 4 results.
## Use doi instead.
# preprint_source_df = (
#     source_lf
#     .filter(
#         pl.col("primary_location_id").is_in(
#             preprint_primary_location_ids.values()
#         )
#     )
#     .collect(engine="streaming")
#     .with_columns(
#         pl.col("primary_location_id").replace(
#             {v: k for k, v in preprint_primary_location_ids.items()}
#         ).alias("server")
#     )
# )

# preprint_source_df.group_by("server").count().sort("count", descending=True)
#
# server	count
# str	u32
# "arxiv"	2074954
# "biorxiv"	281013
# "medrxiv"	66341
# "PsyArxiv"	10
# "osf"	5
# "SocArxiv"	4


In [ ]:
schema = {"doi":pl.String, "primary_category":pl.String, "server":pl.String, "curated_doi":pl.String, "published_date":pl.Datetime}

preprints = (
    pl.concat([
        pl.scan_ndjson(f"{data_path}/arxiv/arxiv_preprints.dedup_0129.jsonl", schema=schema).with_columns(pl.col("server").fill_null("arxiv")),
        pl.scan_ndjson(f"{data_path}/osf/osf_preprints.dedup.jsonl",schema=schema).with_columns(pl.col("server").fill_null("osf")),
        pl.scan_ndjson(f"{data_path}/biorxiv/biorxiv_preprints.dedup.jsonl",schema=schema).with_columns(pl.col("server").fill_null("biorxiv")),
        pl.scan_ndjson(f"{data_path}/medrxiv/medrxiv_preprints.dedup.jsonl", schema=schema).with_columns(pl.col("server").fill_null("medrxiv")),
    ])
    .with_columns(
        pl.col("doi")
        .str.to_lowercase()
        .str.replace_all(r"https://doi.org/", "", literal=True)
    )
    .filter(pl.col("server").is_in(["arxiv","biorxiv","medrxiv","osf","psyarxiv","socarxiv"]))
)

preprints.head().collect(engine="streaming")

In [ ]:
preprints_opal = (
    preprints
    .join(doi_lf_enhanced_arxiv,on="doi",how="left")
    .with_columns(
        pl.col("published_date").dt.year().alias("publication_year_preprint")
    )
)

preprints_opal.sink_parquet("./data/preprints_opal_0129.parquet")

preprints_opal.collect_schema()

In [ ]:
preprints_opal.filter(pl.col('work_id')=="W4417236467").collect(engine="streaming")

In [ ]:
preprints.filter(pl.col("published_date").is_null()).collect(engine="streaming")

In [ ]:
stats_df = (
    preprints_opal.group_by("server").agg(
        pl.col("work_id").is_null().sum().alias("missing_openalex_id"),
        pl.col("work_id").is_not_null().sum().alias("mapped_openalex_id"),
    )
    .with_columns(
        (pl.col("missing_openalex_id") + pl.col("mapped_openalex_id") ).alias("total")
    )
    .with_columns(
        (pl.col("mapped_openalex_id") / pl.col("total")).alias("ratio"),
    )
    .sort("total", "ratio", descending=True)
    .collect(engine="streaming")
)


In [ ]:
print(stats_df.head(6).sum())
stats_df.head(6)

# preperint citing works by sources 

In [ ]:
ref_lf = pl.scan_parquet(f"{openalex_path}/works/referenced_works/*")
preprints_opal = pl.scan_parquet("./data/preprints_opal_0129.parquet")

citing_preprints_lf  = ref_lf.explode("referenced_works").join(
    preprints_opal,
    left_on="referenced_works",
    right_on="work_id",
    how="inner"
)

citing_preprints_lf.sink_parquet("data/citing_preprints_enhanced.parquet")

citing_preprints_df = (
    citing_preprints_lf
    .join(source_lf,on="work_id",how="inner")
    .join(year_lf,on="work_id",how="inner")
    .collect(engine="streaming")
)

citing_preprints_df.write_parquet("./data/citing_preprints_full_enhanced.parquet")

# Impute Curation records

In [ ]:
# Sampled work that has no curation notice in arXiv but has corresponding journal/conference publication on OpenAlex.
# https://openalex.org/works/W2108850836: Aligned Interference Neutralization and the Degrees of Freedom of the 2$\,\times \,$2$\,\times \,$2 Interference Channel IEEE
# https://openalex.org/works/W2949683706: Aligned Interference Neutralization and the Degrees of Freedom of the 2x2x2 Interference Channel

# https://api.openalex.org/works?page=1&filter=best_oa_location.source.id:'https://openalex.org/S2764455111'&per_page=10&include_xpac=true&mailto=zhmuler@gmail.com
# https://api.openalex.org/works?page=1&filter=best_oa_location.source.id:'S2764455111'&per_page=10&include_xpac=true&mailto=zhmuler@gmail.com
# https://api.openalex.org/works?page=1&filter=best_oa_location.source.id:S2764455111&per_page=10&include_xpac=true&mailto=zhmuler@gmail.com
# https://api.openalex.org/works?page=1&filter=best_oa_location.source.id:s4306400194,type:article&per_page=10&include_xpac=true&mailto=zhmuler@gmail.com
# https://api.openalex.org/works?page=1&filter=indexed_in:arxiv,type:article&per_page=10&include_xpac=true&mailto=zhmuler@gmail.com

# candidate filters to try:
# open_access.oa_url == https://arxiv.org/pdf/0904.2132
# locations.id == pmh:oai:arXiv.org:0904.2132
# locations.landing_page_url == pmh:oai:arXiv.org:0904.2132
# best_oa_location.id

In [ ]:
num_part = 40
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as sklearn_sw

lf_title = pl.scan_parquet(openalex_path+ "/works/display_name/*").with_columns(pl.col('display_name').str.to_lowercase())

def tok(col: str) -> pl.Expr:
    return (
        pl.col(col)
        .str.replace_all(r"[^\p{L}]+", " ")
        .str.strip_chars()
        .str.split(" ")
        .list.eval(
            pl.element()
            .filter(
                pl.element().str.len_chars() >= 2,
            )
        )
    )

tokenized_title = (
    lf_title
    .with_row_index("row_idx")
    .with_columns(
        tok("display_name").alias("tokens"),
        (pl.col("row_idx")%num_part).alias("part_idx"),
    )
)

for i in range(num_part):
    (
        tokenized_title
        .filter(pl.col("part_idx")==i)
        .select("work_id", "tokens")
        .sink_parquet(openalex_path+ f"/works/tokenized_display_name_inclusive/part_{i:04d}.parquet.zstd", compression="zstd")
    )

In [ ]:
import polars as pl
from constant import openalex_path
lf_tokens = pl.scan_parquet(openalex_path+ "/works/tokenized_display_name/*")

lf_first_authors = pl.scan_parquet(openalex_path+ "/works/author_id_grpby/*").with_columns(pl.col("author_id").list.first())
lf_author_count = lf_first_authors.group_by("author_id").len()
lf_first_authors_valid_size = lf_first_authors.join(
    lf_author_count.filter(pl.col("len")<300), #roughly 1 paper per month for 30 years or 5 paper per year for 60 years
    on="author_id",
    how='semi'
)

preprints_opal = pl.scan_parquet("./data/preprints_opal_0129.parquet")

In [ ]:
# logarithmic binning for author count distribution

lf_author_count.group_by(
    pl.col("len").log10().floor().alias("log_bin")
).len().collect(engine="streaming")

In [ ]:
no_curation_record = (
    preprints_opal
    .filter(pl.col("curated_doi").is_null())
    .join(lf_tokens, on="work_id", how="left")
    .join(lf_first_authors_valid_size, on="work_id", how="left")
    .drop_nulls(subset=["author_id", "tokens"])
)
no_curation_record.sink_parquet("./data/no_curation_record_preprints_300papers.parquet")

# direct match


In [ ]:
frequnet_tokens = (
    lf_tokens
    .head(100)
    .explode("tokens")
    .group_by("tokens")
    .len()
    .sort("len", descending=True)
    .collect(engine="streaming")
)
frequnet_tokens

In [ ]:
num_part = 50
no_curation_record = pl.scan_parquet("./data/no_curation_record_preprints_300papers.parquet")
candidates_article_first_author = (
    no_curation_record
    .join(lf_first_authors_valid_size, on="author_id", how="left")
    .join(lf_tokens.drop_nulls(), left_on="work_id_right", right_on="work_id", how="left")
    .with_row_index("row_idx")
    .with_columns(
        (pl.col("row_idx")%num_part).alias("part_idx"),
    )
    .select("work_id","tokens","work_id_right","tokens_right","part_idx")
    .rename({
        "work_id":"preprint_work_id",
        "work_id_right":"article_work_id",
        "tokens":"preprint_tokens",
        "tokens_right":"article_tokens",
    })
)

candidates_scored = (
    candidates_article_first_author
    .with_columns([
        pl.col("preprint_tokens").list.set_intersection(pl.col("article_tokens")).list.len().alias("inter"),
        pl.col("preprint_tokens").list.len().alias("len_a"),
        pl.col("article_tokens").list.len().alias("len_b"),
    ])
    .filter(
        pl.col("len_a") > 0,
        pl.col("len_b") > 0,
        pl.col('preprint_work_id') != pl.col('article_work_id')
    )
    .with_columns(
        pl.when((pl.col("len_a") + pl.col("len_b")) > 0)
        .then(2 * pl.col("inter") / (pl.col("len_a") + pl.col("len_b")))
        .otherwise(0.0)
        .alias("dice")
    )
    .filter(pl.col("dice") >= 0.1)
)

for i in range(num_part):
    (
        candidates_scored
        .filter(pl.col("part_idx")==i)
        .drop("part_idx")
        .sink_parquet(f"~/r2/arxiv/candidates_article_first_author_300papers/part_{i:04d}.parquet.zstd", compression="zstd")
    )

# matched

In [ ]:
preprint_opal = pl.scan_parquet("./data/preprints_opal_0129.parquet")

df = (
    pl.scan_parquet("~/r2/arxiv/candidates_article_first_author_300papers/part_*.parquet.zstd")
    .join(preprint_opal.select("work_id"), left_on="article_work_id", right_on="work_id", how="anti")
    .join(doi_lf, left_on="article_work_id", right_on="work_id", how="left")
    .drop_nulls(subset=["doi"])
    .unique(subset="preprint_work_id", keep="first")
)

In [ ]:
import numpy as np  

df_cut = (
    df
    .with_columns(
        pl.col("dice")
        .cut(
            breaks=list(np.linspace(0,1,11) * 100 // 10 / 10),
            left_closed=False
        )
        .alias("dice_bin")
    )
)

hist = df_cut.group_by("dice_bin").len().collect(engine='streaming').sort("dice_bin", descending=True)

In [ ]:
# visualize df_cut the distribution of dice score for the candidate pairs use matplotlib or seaborn
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.bar(hist.sort('dice_bin')["dice_bin"], hist.sort('dice_bin')["len"])
plt.title("Distribution of Dice Scores for Candidate Pairs")
plt.grid(alpha=0.3)
plt.tight_layout()  



In [ ]:
hist.with_columns(pl.col('len').cum_sum().alias("cum_count"))

In [ ]:
imputed = (
        df
        .filter(pl.col("dice")>0.3)
        .select("preprint_work_id", "article_work_id", "dice")
        .join(doi_lf.select(["work_id", "doi"]), left_on="article_work_id", right_on="work_id", how="left")
        .select(
            pl.col("preprint_work_id"),
            pl.col("article_work_id"),
            pl.col("doi").alias("imputed_doi"),
            pl.col("dice"),
        )
    )

imputed.filter(pl.col("imputed_doi").is_null()).collect(engine="streaming")

In [ ]:
imputed.filter(pl.col("imputed_doi").is_not_null()).head().collect(engine="streaming")

In [ ]:
preprint_opal.head().collect(engine="streaming")

In [ ]:
imputed.join(
    preprint_opal.filter(pl.col("curated_doi").is_null()),
    left_on="preprint_work_id",
    right_on="work_id",
    how="left"
).filter(pl.col("imputed_doi").is_not_null()).count().collect(engine="streaming")

In [ ]:
preprint_opal = pl.scan_parquet("./data/preprints_opal_0129.parquet")
preprint_opal.head().collect(engine="streaming")

preprint_opal_curation_imputed = (
    preprint_opal
    .join(imputed, left_on="work_id", right_on="preprint_work_id", how="left")
    # curated_doiが Noneのところに、新たに imputed_doi からdoiを埋める
    .with_columns(
        pl.when(pl.col("curated_doi").is_null())
        .then(pl.col("imputed_doi"))
        .otherwise(pl.col("curated_doi"))
        .alias("curated_doi")
    )
)

In [ ]:
preprint_opal_curation_imputed = pl.scan_parquet("./data/preprints_opal_0129_curation_imputed.parquet")
preprint_opal_curation_imputed.group_by(pl.col("curated_doi").is_not_null()).len().collect(engine="streaming")        

In [ ]:
preprint_opal_curation_imputed.sink_parquet("./data/preprints_opal_0129_curation_imputed.parquet")